# Tools

Um modelo de linguagem produz tokens, e nada além disso. Toda ação sobre o mundo, calcular com exatidão, ler um arquivo, consultar um serviço, acontece fora dele, em código que alguém escreveu. Ferramenta é o nome do arranjo que liga as duas coisas: o modelo escreve um pedido em formato conhecido, o programa executa a função correspondente e devolve o resultado no contexto, e o modelo continua a resposta já com o valor em mãos.

O notebook começa mostrando o que o modelo não consegue fazer sozinho, examina o formato de chamada que os modelos ajustados para ferramentas já trazem no template, constrói o decorador que gera o esquema a partir da assinatura da função, fecha o laço de chamar e observar, e termina com três ferramentas de naturezas diferentes: uma que calcula, uma que escreve e lê arquivo, e uma que consulta um serviço externo.

In [ ]:
import inspect
import json
import urllib.request
from pathlib import Path
from typing import Callable, get_type_hints

import pandas as pd
import torch

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=200)
print(llm.model)

## O limite do texto

As duas células seguintes pedem coisas que parecem simples. Antes de rodar, tente prever o resultado de cada uma.

In [ ]:
print(llm.invoke([{"role": "user", "content": "How much is 4871 times 3926? Answer with the number only."}]))
print(f"resposta correta: {4871 * 3926}")

A multiplicação exata exige um algoritmo com transporte entre casas, e o que o modelo faz é prever tokens plausíveis para um número dessa forma. O resultado sai com a quantidade certa de dígitos e o valor errado.

In [ ]:
print(llm.invoke([
    {"role": "user", "content": "Save a one-line summary of the Python language to a file named resumo.txt."},
], max_tokens=100))
print(f"arquivo existe: {Path('resumo.txt').exists()}")

O modelo devolve o código que resolveria o pedido e o arquivo não existe, porque nada foi executado. Os dois casos têm a mesma causa: a saída é texto, e texto não multiplica nem grava em disco.

## Formato de ferramentas

O caminho para resolver os dois é dar ao modelo um vocabulário para pedir a execução de uma função, e um lugar para receber o resultado. Modelos ajustados para ferramentas já trazem esse vocabulário no template de conversa.

### Esquema da ferramenta

A ferramenta é apresentada ao modelo como uma declaração, com o nome, uma descrição em linguagem natural e os parâmetros descritos em JSON Schema. Essa declaração é o que o modelo lê, e é por ela que ele decide se e como chamar.

In [ ]:
multiply_schema = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    },
}

### O prompt com ferramentas

O `apply_chat_template` recebe as declarações no argumento `tools` e as insere no prompt. O resultado é uma string, como qualquer outro prompt.

In [ ]:
question = [{"role": "user", "content": "How much is 4871 times 3926?"}]
prompt = llm.tokenizer.apply_chat_template(
    question, tools=[multiply_schema], tokenize=False, add_generation_prompt=True
)
print(prompt)

O template escreveu a declaração dentro de um bloco `<tools>` na mensagem de sistema e acrescentou a instrução de responder com um objeto JSON entre `<tool_call>` e `</tool_call>`. O protocolo inteiro é texto convencionado, aprendido pelo modelo durante o ajuste, e cada família de modelos usa marcadores próprios.

### A chamada emitida pelo modelo

Com as ferramentas no prompt, a geração segue igual às anteriores.

In [ ]:
answer = llm.generate(prompt, max_tokens=120)
print(answer)

O modelo escolheu a ferramenta, preencheu os argumentos e parou. Nada foi executado: o que existe é uma string com a intenção de chamada, e transformá-la em execução é responsabilidade do programa que fez a chamada.

### A mensagem de papel tool

O resultado da execução volta ao modelo como uma mensagem nova, com o papel `tool`. O turno do assistente que pediu a chamada também precisa aparecer, para que o histórico registre o que foi pedido.

In [ ]:
conversation = question + [
    {"role": "assistant", "content": "", "tool_calls": [
        {"type": "function", "function": {"name": "multiply", "arguments": {"a": 4871, "b": 3926}}},
    ]},
    {"role": "tool", "name": "multiply", "content": str(4871 * 3926)},
]
with_result = llm.tokenizer.apply_chat_template(
    conversation, tools=[multiply_schema], tokenize=False, add_generation_prompt=True
)
print(with_result[with_result.index("<|im_start|>user") :])

O papel `tool` existe na lista de mensagens e não no prompt: este template o converte em um turno de usuário com o conteúdo dentro de `<tool_response>`. A lista de dicionários é a estrutura de programação, e o texto renderizado é o que o modelo recebe.

In [ ]:
print(llm.generate(with_result, max_tokens=60))

A resposta final usa o número que veio da execução. O ciclo completo tem quatro passos: declarar, pedir, executar, devolver.

## O decorador tool

Escrever o esquema à mão duplica o que a função já declara. Nome, parâmetros e tipos estão na assinatura, e a descrição está na docstring. O decorador lê essas informações por introspecção e anexa o esquema à própria função.

In [ ]:
TYPE_NAMES = {str: "string", int: "integer", float: "number", bool: "boolean"}


def tool(fn: Callable) -> Callable:
    """Anexa fn.tool_schema por introspecção e devolve a própria função."""
    description = inspect.getdoc(fn)
    if not description:
        raise ValueError(f"A ferramenta {fn.__name__} precisa de docstring.")
    hints = get_type_hints(fn)
    properties = {
        name: {"type": TYPE_NAMES[hints[name]]}
        for name in inspect.signature(fn).parameters
    }
    fn.tool_schema = {
        "type": "function",
        "function": {
            "name": fn.__name__,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
            },
        },
    }
    return fn

O decorador recusa função sem docstring, porque a descrição é o que o modelo lê para decidir. A tabela de tipos aceita quatro entradas de propósito: um parâmetro de tipo não previsto levanta `KeyError` na definição da ferramenta, e não silenciosamente em produção.

A docstring das ferramentas fica em inglês, porque ela entra no prompt. Os comentários e as docstrings do resto do código continuam em português.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as 12 * (3 + 4)."""
    # O modelo escreve essa string, então a avaliação roda sem acesso a builtins.
    return str(eval(expression, {"__builtins__": {}}, {}))

In [ ]:
print(json.dumps(calculate.tool_schema, indent=2))

O decorador devolve a própria função, que continua chamável e testável sem modelo nenhum. Essa é a razão de anexar o esquema em vez de embrulhar a função em outro objeto.

In [ ]:
print(calculate("4871 * 3926"))

A avaliação acontece sem builtins, o que impede o acesso a `open`, a `__import__` e ao resto da biblioteca padrão a partir da string. É a proteção mínima para executar algo que veio do modelo, e ainda assim uma expressão gigante trava o processo. Uma ferramenta de produção usaria um analisador de expressões em vez de `eval`.

## O laço de chamada

Três funções fecham o ciclo: uma que extrai a chamada do texto, uma que executa a ferramenta pedida e uma que alterna as duas até o modelo responder sem chamar nada.

In [ ]:
def parse_tool_call(text: str) -> dict | None:
    """Extrai a chamada do texto do modelo e devolve None quando é resposta final."""
    if "<tool_call>" not in text:
        return None
    block = text.split("<tool_call>")[1].split("</tool_call>")[0]
    try:
        call = json.loads(block)
    except json.JSONDecodeError:
        return None
    return call if isinstance(call.get("name"), str) else None

In [ ]:
print(parse_tool_call(answer))
print(parse_tool_call("The product is 19,123,546."))

O `None` é o sinal de que o texto é a resposta final, e é ele que encerra o laço.

In [ ]:
def run_tool(call: dict, tools: list[Callable]) -> str:
    """Executa a ferramenta pedida e devolve o resultado como texto."""
    for fn in tools:
        if fn.tool_schema["function"]["name"] == call["name"]:
            try:
                return str(fn(**call["arguments"]))
            except Exception as error:
                # A exceção vira observação, porque o modelo precisa poder reagir a ela.
                return f"{type(error).__name__}: {error}"
    return f"unknown tool: {call['name']}"

Esta é a única captura ampla de exceção do notebook, e ela existe porque um erro de ferramenta é caso normal do laço e não motivo para interromper a execução. Ferramenta inexistente recebe o mesmo tratamento, já que o nome também vem do modelo.

In [ ]:
def run_with_tools(question: str, tools: list[Callable], max_steps: int = 3) -> list[dict]:
    """Alterna chamada e execução até a resposta final e devolve o histórico completo."""
    messages = [{"role": "user", "content": question}]
    schemas = [fn.tool_schema for fn in tools]
    for _ in range(max_steps):
        prompt = llm.tokenizer.apply_chat_template(
            messages, tools=schemas, tokenize=False, add_generation_prompt=True
        )
        text = llm.generate(prompt)
        call = parse_tool_call(text)
        if call is None:
            return messages + [{"role": "assistant", "content": text}]
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": call},
        ]})
        messages.append({"role": "tool", "name": call["name"], "content": run_tool(call, tools)})
    return messages + [{"role": "assistant", "content": "step limit reached"}]

O limite de passos é a primeira proteção contra laço infinito, e devolver o histórico inteiro em vez da resposta final deixa o traço da execução disponível para leitura.

In [ ]:
history = run_with_tools("How much is 4871 times 3926?", [calculate])
pd.DataFrame([
    {"role": message["role"], "content": message.get("content", "")[:70],
     "tool_calls": json.dumps(message.get("tool_calls", ""))[:60]}
    for message in history
])

In [ ]:
print(history[-1]["content"])

A falha da abertura está resolvida, e o número da resposta veio de uma multiplicação em Python.

## Casos de uso

As três ferramentas seguintes têm naturezas diferentes, e a diferença aparece no risco de cada uma. A primeira só calcula. A segunda muda o estado do disco. A terceira depende de um serviço que pode estar fora do ar.

### Arquivos

Ferramenta com efeito colateral precisa de fronteira. As duas abaixo trabalham dentro de um diretório de trabalho, e nomes que tentem sair dele são recusados antes de qualquer acesso.

In [ ]:
WORKSPACE = Path("workspace")


def resolve(name: str) -> Path:
    """Resolve o nome dentro do diretório de trabalho e recusa caminhos fora dele."""
    WORKSPACE.mkdir(exist_ok=True)
    target = (WORKSPACE / name).resolve()
    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("Path outside the workspace")
    return target

In [ ]:
@tool
def write_file(name: str, content: str) -> str:
    """Write text content to a file in the workspace."""
    path = resolve(name)
    path.write_text(content, encoding="utf-8")
    return f"wrote {len(content)} characters to {name}"


@tool
def read_file(name: str) -> str:
    """Read a file from the workspace and return its content. Use this tool whenever the user asks what a file contains."""
    return resolve(name).read_text(encoding="utf-8")

In [ ]:
history = run_with_tools(
    "Save a one-line summary of the Python language to a file named resumo.txt.",
    [write_file, read_file],
)
print(history[-1]["content"])

In [ ]:
print(WORKSPACE.joinpath("resumo.txt").read_text(encoding="utf-8"))

O arquivo existe no disco e o conteúdo foi escrito pelo modelo. É a segunda falha da abertura resolvida, e a primeira vez que uma resposta do modelo produz efeito fora do processo.

In [ ]:
history = run_with_tools("What is in the file resumo.txt?", [write_file, read_file])
print(history[-1]["content"])

A leitura passou pela mesma função de fronteira, e o conteúdo entrou na resposta como observação.

### API externa

A ferramenta abaixo consulta um serviço público de previsão do tempo, sem chave de acesso. Ela traz para o contexto um dado que não existe nos pesos do modelo e que muda a cada hora.

In [ ]:
@tool
def get_temperature(latitude: float, longitude: float) -> str:
    """Get the current temperature in Celsius for a latitude and longitude."""
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    with urllib.request.urlopen(url, timeout=10) as response:
        data = json.load(response)
    return f"{data['current']['temperature_2m']} C"

In [ ]:
history = run_with_tools(
    "What is the current temperature in Natal, Brazil? Its coordinates are -5.79, -35.21.",
    [get_temperature],
)
print(history[-2]["content"])
print(history[-1]["content"])

A penúltima mensagem é a observação crua devolvida pelo serviço, e a última é a resposta escrita a partir dela. Uma ferramenta assim acrescenta três problemas que as anteriores não têm: latência variável, falha fora do controle do programa e resposta que muda entre execuções, o que impede comparar duas rodadas por igualdade de texto.

## Escolha e recuperação

Ter a ferramenta disponível não garante que ela seja usada, nem que seja usada na hora certa.

### A descrição decide a escolha

A mesma pergunta é feita duas vezes, com a mesma função e duas descrições diferentes.

In [ ]:
VAGUE = "Read the text content of a file in the workspace directory."
for description in [VAGUE, read_file.tool_schema["function"]["description"]]:
    schema = json.loads(json.dumps(read_file.tool_schema))
    schema["function"]["description"] = description
    prompt = llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is in the file resumo.txt?"}],
        tools=[schema], tokenize=False, add_generation_prompt=True,
    )
    print(repr(description[:40]), "->", repr(llm.generate(prompt, max_tokens=60)[:70]))

Com a descrição vaga o modelo responde que precisaria ler o arquivo, sem pedir a leitura. Com a descrição que diz quando usar a ferramenta, ele chama. A descrição faz parte do prompt e se escreve com o mesmo cuidado de uma instrução.

### Chamada desnecessária

O erro simétrico é chamar ferramenta quando a pergunta não pede nenhuma.

In [ ]:
history = run_with_tools("Good morning! How are you today?", [calculate, write_file, read_file])
pd.DataFrame([{"role": message["role"], "content": message.get("content", "")[:60]} for message in history])

O histórico tem duas mensagens e nenhuma chamada, que é o comportamento correto. Esse caso não é garantido: quanto mais ferramentas disponíveis, maior a chance de o modelo usar uma sem necessidade, e por isso o conjunto de ferramentas se mantém pequeno e específico.

### Erro de ferramenta como observação

A pergunta abaixo pede um arquivo que não existe.

In [ ]:
history = run_with_tools("Read the file inexistente.txt and tell me what it says.", [read_file])
for message in history[1:]:
    print(message["role"], ":", message.get("content", "")[:120])

A exceção virou texto na mensagem de papel `tool`, o laço continuou e o modelo relatou a falha em vez de inventar um conteúdo. Erro tratado como observação é o que permite ao modelo mudar de rota, e é também o que evita que uma falha de ferramenta derrube a execução inteira.

O `agentkit` traz esse conjunto em `tools.py`, com o mesmo decorador e as funções de leitura e execução da chamada. A diferença é que lá o protocolo é escrito à mão em um prompt de sistema, em vez de vir do template, o que mantém o pacote utilizável com modelos que não foram ajustados para ferramentas.

## Exercícios

### Exercício 1

Escreva uma ferramenta que devolva a data e a hora atuais e responda com ela a pergunta "que dia é hoje". Depois faça a mesma pergunta sem oferecer a ferramenta e compare as duas respostas.

In [ ]:
def current_datetime() -> str:
    ...

### Exercício 2

Escreva uma ferramenta de busca por palavra-chave sobre a lista de documentos abaixo, que receba um termo e devolva os trechos que o contêm. Pergunte algo que só possa ser respondido com o conteúdo dessa lista.

In [ ]:
DOCUMENTS = [
    "Refunds are issued within 5 business days after approval.",
    "Plan changes take effect on the next billing cycle.",
    "Accounts inactive for 12 months are archived automatically.",
    "Support hours are from 9am to 6pm on business days.",
]

### Exercício 3

Ofereça `calculate` e `get_temperature` na mesma chamada e faça três perguntas: uma de cálculo, uma de temperatura e uma que não precisa de ferramenta. Monte uma tabela com a pergunta, a ferramenta escolhida e se a escolha foi correta.

In [ ]:
questions = []

### Exercício 4

Peça ao modelo que escreva um arquivo com um nome que tente sair do diretório de trabalho, como `../roubado.txt`, e mostre o que a fronteira devolveu ao laço. Diga o que o modelo fez com essa observação.

In [ ]:
escape_request = ""

### Exercício 5

Escreva uma ferramenta que dependa de um parâmetro de tipo não previsto em `TYPE_NAMES`, como uma lista, e observe onde o erro aparece. Depois estenda a tabela de tipos para aceitá-lo e refaça a chamada.

In [ ]:
def summarize_numbers(values: list) -> str:
    ...